# Phase 18: Does Topology Improve Localization, Not Just Classification?

Every prior topology ablation in this project (Phase 6c, 11) tested per-residue
**classification** (PR-AUC): does knowing about persistent homology help decide
*which residues* are binding sites. That question is now answered, yes,
properly powered, p<1e-5.

This asks a different, untested question: does topology help the model
produce a more accurate **single predicted point**, the trained localization
head added in Phase 14? This is not a re-run of the old ablation on a new
metric; it is a genuinely distinct capability, since Phase 8 already showed
these two heads decouple (training the localization head changed what the
*classification* head found useful, in an unexpected direction). There is no
guarantee topology's contribution to one transfers to the other.

**Why this hasn't been tested by anyone else, as far as the literature check
in this project found:** the persistent-homology-for-protein-binding
literature (Cang & Wei's line of work, PATH/PATH+, persistent Laplacians) uses
classical ML (gradient boosting, SVMs) for **affinity regression**, never a
trained equivariant localization head. EquiPocket and VN-EGNN use equivariant
GNNs for exactly this localization task, but neither uses topological
features at all. This sits in the gap between the two literatures.

### Design: reuse everything expensive, train only what's new

- `full` (topology present): **reuses Phase 15's already-trained checkpoints**
 , no retraining, no risk of introducing a confound from a different training
  run.
- `no_topo` (topology zeroed, same architecture/budget): a **new** 3-seed
  ensemble, trained here.
- Both are scored on **Phase 15's cached benchmark features**, no
  re-fetching, no re-running the ~170-minute featurization. This whole
  experiment costs one training run (~15 min) plus fast inference.

## Cell 1: Setup

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84

In [ ]:
import pickle

import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase10_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

## Cell 2: Recover true ligand centers for training data (Phase 14 logic, reused)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_LIG_ATOMS = 6
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def ligands_and_ca(pid, path, chain_id, n_expect):
    from Bio.PDB import MMCIFParser
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[chain_id]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues) != n_expect: return None
    raw_ca = np.asarray([r["CA"].coord for r in residues], dtype=np.float32)

    ligs = []
    for c2 in model:
        for res in c2:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                             dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    return raw_ca, ligs

t0=time.time(); n_ok=0; n_geom_fail=0; n_no_ligand=0
for d in dataset:
    if d.pdb_id not in paths: continue
    r = ligands_and_ca(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if r is None: continue
    raw_ca, ligs = r
    if not ligs: n_no_ligand += 1; continue

    # geometry-drift check: pairwise CA distances must match d.pos (residue ORDER, not just count)
    k = min(50, len(raw_ca))
    a = np.linalg.norm(raw_ca[:k,None]-raw_ca[None,:k], axis=-1)
    b = np.linalg.norm(d.pos.numpy()[:k,None]-d.pos.numpy()[None,:k], axis=-1)
    if float(np.abs(a-b).max()) > 1e-2:
        n_geom_fail += 1; continue

    offset = raw_ca.mean(axis=0)                       # same centering d.pos already used
    y_np = d.y.numpy().astype(bool)
    contact_centroid = raw_ca[y_np].mean(axis=0) if y_np.any() else raw_ca.mean(axis=0)
    # "primary" ligand = the one whose center of mass is nearest the labelled contact residues
    coms = [lig.mean(axis=0) for lig in ligs]
    best = int(np.argmin([np.linalg.norm(c - contact_centroid) for c in coms]))
    d.true_center = torch.from_numpy((coms[best] - offset).astype(np.float32))
    n_ok += 1

print(f"\ntrue centers recovered: {n_ok:,}/{len(dataset):,} "
      f"({(time.time()-t0)/60:.1f} min) | geometry mismatches: {n_geom_fail} | no ligand: {n_no_ligand}")

dataset = [d for d in dataset if hasattr(d, "true_center")]
print(f"training chains with recovered true centers: {len(dataset):,}")

rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
n_tr, n_va = len(splits["train"]), len(splits["val"])
print(f"deployment training: {n_tr} train / {n_va} val")

## Cell 3: Model (identical to Phase 14/15/16)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 4: Training machinery (identical to Phase 14)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 5: Train the `no_topo` ensemble (3 seeds, lambda=0.1, topology zeroed)

`lambda=0.1` reused from Phase 14's sweep, not re-tuned, this experiment
isolates the topology-vs-no-topology question, and re-tuning lambda separately
for each arm would confound that comparison with a second variable.

In [ ]:
import time
WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

ORIG_X = [d.x.clone() for d in dataset]
def zero_topology():
    for d, x0 in zip(dataset, ORIG_X):
        t = x0[:, N_PRE:].clone(); t.zero_()
        d.x = torch.cat([x0[:, :N_PRE], t], dim=1)
def restore_topology():
    for d, x0 in zip(dataset, ORIG_X):
        d.x = x0.clone()

zero_topology()
BEST_LAMBDA = 0.1
notopo_ensemble = []
t0 = time.time()
for seed in (0,1,2):
    m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
    v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
    notopo_ensemble.append(m)
    torch.save(m.state_dict(), WORK/f"cache/notopo_deploy_seed{seed}.pt")
    val_dcc = v.get("dcc", float("nan"))
    print(f"  seed {seed}: val PR {val_pr:.4f}  val DCC {val_dcc:.4f}  [{(time.time()-t0)/60:.1f}m]")
restore_topology()
print(f"no_topo ensemble of {len(notopo_ensemble)} trained on {len(dataset):,} chains (topology zeroed)")

## Cell 6: Load Phase 15's `full` checkpoints (no retraining) + Phase 15's cached benchmark features

In [ ]:
full_ensemble = []
for seed in (0,1,2):
    cands = []
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): cands.extend(r.rglob(f"loc_deploy_seed{seed}.pt"))
    state = torch.load(max(cands, key=lambda p: p.stat().st_size), map_location=DEVICE)
    m = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE); m.load_state_dict(state); m.eval()
    full_ensemble.append(m)
print(f"loaded {len(full_ensemble)} 'full' checkpoints from Phase 15 (no retraining)")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

with open(locate("phase15_raw_features.pkl"), "rb") as fh:
    raw_cache = pickle.load(fh)
blob6b_bench = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
mu_b, sd_b = blob6b_bench["topo_mean"], blob6b_bench["topo_std"]
print(f"loaded cached features for {len(raw_cache):,} benchmark chains")

## Cell 7: Score both arms on the SAME cached benchmark chains, frame-corrected

In [ ]:
from collections import defaultdict

@torch.no_grad()
def predict_chain(entry, model_ensemble, zero_topo=False):
    x = torch.from_numpy(entry["x"]).clone()
    if zero_topo:
        x[:, N_PRE:] = 0.0
    else:
        x[:, N_PRE:] = (x[:, N_PRE:] - torch.from_numpy(mu_b)) / torch.from_numpy(sd_b)
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=torch.from_numpy(entry["pos"]).to(DEVICE)
    d.edge_index=torch.from_numpy(entry["edge_index"]).to(DEVICE)
    d.edge_attr=torch.from_numpy(entry["edge_attr"]).to(DEVICE)
    d.batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
    seed_logits, seed_centers = [], []
    for m in model_ensemble:
        lg, pc = m(d)
        seed_logits.append(torch.sigmoid(lg).cpu().numpy())
        seed_centers.append(pc.cpu().numpy()[0])
    probs = np.mean(seed_logits, axis=0)
    offset = entry["ca"].mean(axis=0)
    center_raw = np.mean(seed_centers, axis=0) + offset
    return probs, center_raw


def dcc_dca(pred_center, ligs):
    best_dcc, best_dca = np.inf, np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        com = lig_atoms.mean(axis=0)
        d_ = float(np.linalg.norm(pred_center - com))
        if d_ < best_dcc:
            best_dcc = d_
            best_dca = float(np.linalg.norm(lig_atoms - pred_center, axis=1).min())
    return (best_dcc if np.isfinite(best_dcc) else None, best_dca if np.isfinite(best_dca) else None)


by_structure = defaultdict(list)
for (src_,pid,cid), entry in raw_cache.items():
    by_structure[(src_,pid)].append((cid, entry))

def score_arm(model_ensemble, zero_topo, tag):
    results = {"coach420": [], "holo4k": []}
    for (src_, pid), chain_list in by_structure.items():
        ligs = chain_list[0][1]["ligs"]
        best_conf, best_center = -np.inf, None
        for cid, entry in chain_list:
            probs, center = predict_chain(entry, model_ensemble, zero_topo)
            conf = float(probs.sum())
            if conf > best_conf:
                best_conf, best_center = conf, center
        dcc, dca = dcc_dca(best_center, ligs)
        if dcc is not None: results[src_].append((dcc, dca))

    summary = {}
    all_dcc = []
    for src_ in ("coach420","holo4k"):
        vals = results[src_]
        if not vals: continue
        dcc_s = np.mean([1.0 if d<4.0 else 0.0 for d,_ in vals])
        dca_s = np.mean([1.0 if a<4.0 else 0.0 for _,a in vals])
        summary[src_] = {"n": len(vals), "dcc": float(dcc_s), "dca": float(dca_s)}
        all_dcc.extend([1.0 if d<4.0 else 0.0 for d,_ in vals])
    pooled = float(np.mean(all_dcc)) if all_dcc else float("nan")
    print(f"[{tag}] " + " | ".join(f"{k} DCC={v['dcc']:.4f}(n={v['n']})" for k,v in summary.items())
          + f" | pooled {pooled:.4f}")
    return summary, pooled


full_summary, full_pooled = score_arm(full_ensemble, zero_topo=False, tag="full (with topology)")
notopo_summary, notopo_pooled = score_arm(notopo_ensemble, zero_topo=True, tag="no_topo")

## Cell 8: Verdict

In [ ]:
delta_pooled = full_pooled - notopo_pooled
print(f"\n{'='*70}")
print(f"DOES TOPOLOGY IMPROVE LOCALIZATION (not just classification)?")
print(f"{'='*70}")
print(f"  full (with topology)  pooled DCC: {full_pooled:.4f}")
print(f"  no_topo                pooled DCC: {notopo_pooled:.4f}")
print(f"  delta: {delta_pooled:+.4f}")
for src_ in ("coach420","holo4k"):
    if src_ in full_summary and src_ in notopo_summary:
        d = full_summary[src_]["dcc"] - notopo_summary[src_]["dcc"]
        print(f"  {src_}: full {full_summary[src_]['dcc']:.4f}  no_topo {notopo_summary[src_]['dcc']:.4f}  "
              f"delta {d:+.4f}")

verdict = ("YES -- topology contributes to localization accuracy specifically, "
          "beyond its already-established classification benefit" if delta_pooled > 0.02
          else "NO CLEAR BENEFIT for localization -- topology's contribution appears "
               "specific to classification (which residues), not localization (where exactly)"
          if abs(delta_pooled) <= 0.02 else
          "topology appears to HURT localization -- worth investigating why, "
          "possibly the loc_head backbone competes differently with topological features")
print(f"\nVERDICT: {verdict}")
print("\n(single run, n~1140 pooled structures, not multi-seed significance tested -- "
      "treat as a first, directional answer to a question nobody else has asked)")

json.dump({"full": {"summary": full_summary, "pooled": full_pooled},
           "no_topo": {"summary": notopo_summary, "pooled": notopo_pooled},
           "delta_pooled": delta_pooled},
          open("/kaggle/working/phase18_summary.json", "w"), indent=2)
print("\nsaved -> phase18_summary.json")

---

## Reading this

This is a **single run**, not the multi-seed significance-tested comparison
that made the classification-side topology claim (Phase 6c/11) trustworthy,
treat the direction as a first answer, not a final one. But directionally:

- **A real gap either way is a genuinely new finding.** Nobody in the surveyed
  literature has asked whether topological features help an equivariant GNN
  localize a pocket center specifically, as distinct from classifying which
  residues are near one.
- **If topology helps localization too**, that strengthens the case for fusing
  it into the shared backbone (as this architecture does) rather than treating
  it as classification-only side information.
- **If it doesn't**, that is itself informative: it would suggest topology
  encodes *what* a pocket looks like (useful for classification) without
  encoding *precisely where its center is* (a geometric property topology, by
  construction, only partially captures), a real, defensible, specific claim
  about what this kind of feature is and isn't good for.